# Recformer medium CV — lr1e5_w300 — 3 folds, train only

Настройки фиксированы: seed42, batch4, accumulation4, FP16, 2+2 эпохи. Те же три medium-фолда и pretrained weights.

1. Создайте Kaggle Dataset с общим ZIP и подключите через Add Input.
2. Включите Internet и GPU T4. HF_TOKEN не нужен: веса уже в ZIP.
3. Запустите Save Version → Save & Run All. По окончании скачайте `*_results.zip` из Output и верните в локальный проект.

Этот ноутбук обучает только назначенную настройку. Test не оценивается; общую настройку выберем локально по validation трёх фолдов. Другая GPU может давать отличия FP16/AMP; версии и skipped updates сохраняются.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
LABEL = 'lr1e5_w300'
EXPECTED_BUNDLE_SHA256 = '84e33338f233f55f8c9cbafce2ffafb6614cf2c1c6ea877c31b3b3f462b2a27c'
EXPECTED_INPUT_MANIFEST_SHA256 = 'cb091bd3e4ab465e99344aca23e35ded3e38a8d946c2d16f376ecb0b1e99cce8'
# Attach recformer_medium_cv_inputs_v1.zip as Kaggle Input; Internet ON; Accelerator GPU T4.
WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/tmp/recformer_inputs_v1')
INPUT_ROOT.mkdir(exist_ok=True)
matches = list(Path('/kaggle/input').rglob('recformer_medium_cv_inputs_v1.zip'))
if matches:
    assert len(matches) == 1, f'Multiple matching archives: {matches}'
    INPUT_ZIP = matches[0]
    with INPUT_ZIP.open('rb') as f:
        assert hashlib.file_digest(f, 'sha256').hexdigest() == EXPECTED_BUNDLE_SHA256, 'Input archive checksum mismatch'
    with zipfile.ZipFile(INPUT_ZIP) as z:
        for name in z.namelist():
            assert (INPUT_ROOT / name).resolve().is_relative_to(INPUT_ROOT.resolve()), 'Unsafe archive member'
        z.extractall(INPUT_ROOT)
else:
    # Kaggle may automatically expand ZIP uploads into a Dataset.
    import shutil
    candidates = []
    for p in Path('/kaggle/input').rglob('input_manifest.json'):
        with p.open('rb') as f:
            if hashlib.file_digest(f, 'sha256').hexdigest() == EXPECTED_INPUT_MANIFEST_SHA256:
                candidates.append(p.parent)
    assert len(candidates) == 1, f'Attach the shared input Dataset; found {candidates}'
    shutil.copytree(candidates[0], INPUT_ROOT, dirs_exist_ok=True)
with (INPUT_ROOT / 'input_manifest.json').open('rb') as f:
    assert hashlib.file_digest(f, 'sha256').hexdigest() == EXPECTED_INPUT_MANIFEST_SHA256
print('Verified inputs:', INPUT_ROOT)


In [ ]:
# Separate Python3.11 environment: works even if Kaggle kernel uses Python3.13.
# Does not change notebook kernel packages. Downloads need Internet.
INSTALLER = Path('/tmp/recformer_uv_installer')
if not (INSTALLER / 'uv').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--target', str(INSTALLER), 'uv==0.9.4'], check=True)
uv_env = os.environ.copy()
uv_binary = INSTALLER / 'bin/uv'
if not uv_binary.is_file():
    binaries = [p for p in INSTALLER.rglob('uv') if p.is_file()]
    assert len(binaries) == 1, f'uv executable missing: {binaries}'
    uv_binary = binaries[0]
uv = [str(uv_binary)]
ENV = Path('/tmp/recformer_py311')
if not (ENV / 'bin/python').exists():
    subprocess.run(uv + ['venv', '--python', '3.11', str(ENV)], env=uv_env, check=True)
PYTHON = ENV / 'bin/python'
subprocess.run(uv + ['pip', 'install', '--python', str(PYTHON), '--index-url',
    'https://download.pytorch.org/whl/cu118', 'torch==2.2.2+cu118'], env=uv_env, check=True)
requirements = json.loads((INPUT_ROOT / 'requirements.json').read_text())
subprocess.run(uv + ['pip', 'install', '--python', str(PYTHON), *requirements], env=uv_env, check=True)
subprocess.run([str(PYTHON), '-c', 'import torch,transformers; print(torch.__version__, transformers.__version__); print(torch.cuda.get_device_name()); assert torch.cuda.is_available()'], check=True)


In [ ]:
OUTPUT = WORK / f'recformer_medium_{LABEL}_kaggle_v1'
command = [str(PYTHON), '-u', str(INPUT_ROOT / 'kaggle_runtime.py'),
    '--input', str(INPUT_ROOT), '--output', str(OUTPUT), '--label', LABEL]
subprocess.run(command + ['--check-only'], check=True)
subprocess.run(command, check=True)


In [ ]:
archive = WORK / f'{OUTPUT.name}_results.zip'
assert archive.exists(), 'Result archive missing; inspect earlier errors'
print('Download from Save Version -> Output:', archive.name)
print((OUTPUT / 'completed_folds.json').read_text())
from IPython.display import FileLink, display
display(FileLink(str(archive)))
